# BCO Dew Point Investigation
The Surfacemet WXT instrument records the relative humidity at the station with the variable `RH`.

In [ ]:
import calendar
from pathlib import Path
import textwrap

from matplotlib.lines import Line2D
from matplotlib.patches import Ellipse, Rectangle
import matplotlib.pyplot as plt
from metpy.calc import dewpoint_from_relative_humidity
from metpy.units import units
# from meteocalc import dew_point, Temp
from moist_thermodynamics.moist_thermodynamics.functions import relative_humidity_to_specific_humidity
from moist_thermodynamics.moist_thermodynamics.functions import specific_humidity_to_partial_pressure
from moist_thermodynamics.moist_thermodynamics.saturation_vapor_pressures import es
import numpy as np
import xarray as xr

In [ ]:
import intake

cat = intake.open_catalog("https://tcodata.mpimet.mpg.de/catalog.yaml")
wxt = cat.BCO.surfacemet_wxt_v1.to_dask()

In [ ]:
wxt

In [ ]:
relative_humidity = wxt["RH"].compute()
relative_humidity.plot()

From this initial plot, there are observable gaps in 2013, late 2014, two in late 2015, one in early 2016, one in 2022


In [ ]:
# Check missing values in relative humidity
missing_rh = relative_humidity.isnull().sum()
print(f"Number of missing values in relative humidity: {missing_rh.values}")

## Histogram of Relative Humidity Measurements
The following histogram shows the relative humidity measurements for the entire dataset.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

n, bins, patches = ax.hist(
    relative_humidity.values,
    bins=np.arange(40, 101, 1),
    edgecolor='black'
)

mode_index = np.argmax(n)
mode = bins[mode_index]
mode_frequency = n[mode_index]

ax.set_title("Distribution of all relative humidity measurements at the BCO")
ax.set_xlabel("Relative Humidity / %")
ax.set_ylabel("Frequency")
ax.set_xlim(50, 100)

ax.annotate(
    text=f"Mode: {mode}%",
    xy=(mode + 1, mode_frequency),
    xytext=(mode + 7, mode_frequency),
    arrowprops=dict(
        arrowstyle="->",
        color='red',
        linewidth=2
    )
)
patches[mode_index].set_facecolor("orange")

plt.show()

The distribution as well as the mode is shown in the diagram. The results are as expected.

## Hourly Relative Humidity Analysis
We investigate how the relative humidity is distributed across the hours.

In [ ]:
# relative_humidity = relative_humidity.load() # to prevent ValueError: Aggregation nanquantile

hourly_rh = relative_humidity.groupby("time.hour")

mean_rh = hourly_rh.mean()
max_rh = hourly_rh.max()
min_rh = hourly_rh.min()
p99_rh = hourly_rh.quantile(0.99)
p01_rh = hourly_rh.quantile(0.01)

In [ ]:
xmin, xmax, ymin, ymax = 0, 23, 32, 101
sunrise, sunset = 10, 22

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(max_rh["hour"].values, max_rh, color="red", label="Max RH")
ax.plot(mean_rh["hour"].values, mean_rh, color="black", label="Mean RH")
ax.plot(min_rh["hour"].values, min_rh, color="blue", label="Min RH")
ax.plot(max_rh["hour"].values, p99_rh, color="red", label="99 % RH", alpha=0.5)
ax.plot(max_rh["hour"].values, p01_rh, color="blue", label="1 % RH", alpha=0.5)

ax.axvline(x=sunrise, color="black", linestyle="dashed", linewidth=2)
ax.axvline(x=sunset, color="black", linestyle="dashed", linewidth=2)
ax.axvspan(sunrise, sunset, color="red", alpha=0.15, label="Daytime")

ax.set_xlabel("Hour of Day")
ax.set_ylabel("Relative Humidity / %")
ax.set_title("Ave, Max, and Min Relative Humidity at BCO by hour of day")
ax.set_ylim(ymin, ymax)
ax.set_xlim(xmin, xmax)
ax.set_xticks(np.arange(xmin, xmax + 1, 2))
ax.legend()

plt.show()

The plot produces expected results. Barbados has a maritime tropical climate wherein the consistently high levels of moisture keeps the relative humidity consistently high throughout the day between $80\%$ and $60\%$. In extreme cases, it may drop below $60\%$ while occasional showers can raise it above $90\%$. This pattern is evident in the max and min plots and the percentile plots. Relative humidity also remains high overnight then maximizes at dawn before dropping rapidly as daytime heating occurs - achieving a minimum around noon. This is also evident in the plot, particularly in the `min` and `mean` plots.

 Since this is a subhourly dataset, it is better to look at the daily timeseries first.

## Monthly Relative Humidity Analysis

In [ ]:
# relative_humidity = relative_humidity.load() # to prevent ValueError: Aggregation nanquantile

hourly_rh = relative_humidity.groupby("time.month")

mean_rh = hourly_rh.mean()
max_rh = hourly_rh.max()
min_rh = hourly_rh.min()
p99_rh = hourly_rh.quantile(0.99)
p01_rh = hourly_rh.quantile(0.01)

In [ ]:
xmin, xmax, ymin, ymax = 1, 12, 40, 101
wet_season_start, wet_season_end = 5, 11
ticks = np.arange(xmin, xmax + 1, 1)

fig, ax = plt.subplots(figsize=(10, 8))

ax.plot(max_rh["month"].values, max_rh, color="red", label="Max RH")
ax.plot(mean_rh["month"].values, mean_rh, color="black", label="Mean RH")
ax.plot(min_rh["month"].values, min_rh, color="blue", label="Min RH")
ax.plot(max_rh["month"].values, p99_rh, color="red", label="99 % RH", alpha=0.5)
ax.plot(max_rh["month"].values, p01_rh, color="blue", label="1 % RH", alpha=0.5)

ax.axvline(x=wet_season_start, color="black", linestyle="dashed", linewidth=2)
ax.axvline(x=wet_season_end, color="black", linestyle="dashed", linewidth=2)
ax.axvspan(wet_season_start, wet_season_end, color="red", alpha=0.15, label="Wet Season")

ax.set_xlabel("Month of Year")
ax.set_ylabel("Relative Humidity / %")
ax.set_title("Ave, Max, and Min Relative Humidity at BCO by Month of the Year")
ax.set_ylim(ymin, ymax)
ax.set_xlim(xmin, xmax)
ax.set_xticks(np.arange(xmin, xmax + 1, 1))
ax.set_xticklabels(calendar.month_abbr[1:13], rotation=45)
ax.legend()

plt.show()

The relative humidity is slightly higher during the wet season than during the dry season.

## Relationship with Temperature
Relative humidity has an inverse relationship with temperature. We investigate this.

In [ ]:
temperature = wxt["T"].compute()

In [ ]:
plt.scatter(temperature.values, relative_humidity.values, s=1)

In [ ]:
rain = wxt["R"].compute()

rain_events = rain.where(rain > 0, drop=True)
rain_p50 = rain_events.quantile(0.50)
rain_p95 = rain_events.quantile(0.95)

mask_rain = rain > 0
mask_p50 = rain > rain_p50
mask_p95 = rain > rain_p95

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

rect_mask = (
    (temperature >= 24) &
    (temperature <= 28) &
    (relative_humidity >= 70) &
    (relative_humidity <= 85)
)

sample = slice(None, None, 20)

ax.scatter(
    temperature.where(~rect_mask),
    relative_humidity.where(~rect_mask),
    s=1,
    alpha=0.7,
    color='lightgray'
)
ax.scatter(
    temperature.where(rect_mask)[sample],
    relative_humidity.where(rect_mask)[sample],
    s=1,
    alpha=0.3,
    color="lightgray"
)
ax.scatter(
    temperature.where(mask_rain),
    relative_humidity.where(mask_rain),
    color='#8000ff',
    s=1,
    alpha=0.5
)
ax.scatter(
    temperature.where(mask_p50),
    relative_humidity.where(mask_p50),
    color='#FFAC1C',
    s=1,
    alpha=0.75
)
ax.scatter(
    temperature.where(mask_p95),
    relative_humidity.where(mask_p95),
    color='red',
    s=0.8
)
ax.set_xlim(18, 35)
ax.set_title("Relative humidity (%) vs temperature (°C) scatterplot for the BCO")
ax.set_ylabel("Relative Humidity / %")
ax.set_xlabel("Temperature / °C")

rect = Rectangle(
    xy=(18.4, 46),
    width=5,
    height=17,
    edgecolor="red",
    fill=False
)

ellipse = Ellipse(
    xy=(27, 96),
    width=10,
    height=2,
    edgecolor="purple",
    fill=False,
    angle=-75
)

ax.add_patch(rect)
ax.add_patch(ellipse)

ax.annotate(
    "Strange data",
    xy=rect.get_center(),
    ha="center",
    va="center",
    color="red",
    alpha=0.5
)
ax.annotate(
    "Problematic Area 1",
    xy=(rect.get_center()[0], rect.get_center()[1] + 9.5),
    ha="center",
    va="center",
    color="red",
    alpha=0.5
)
ax.annotate(
    "Problematic Area 2",
    xy=(28, 96),
    xytext=(30, 97),
    ha="center",
    va="center",
    color="purple",
    alpha=0.5,
    arrowprops=dict(
        arrowstyle="->",
        color="purple",
        linewidth=2
    )
)
# legend_elements = [
#     Line2D([0], [0], marker='o', color='w', label='No rain',
#            markerfacecolor='lightgray', markersize=5),
#
#     Line2D([0], [0], marker='o', color='w', label='Rain',
#            markerfacecolor='#8000ff', markersize=5),
#
#     Line2D([0], [0], marker='o', color='w', label='>50th percentile rain',
#            markerfacecolor='#FFAC1C', markersize=6),
#
#     Line2D([0], [0], marker='o', color='w', label='>95th percentile rain',
#            markerfacecolor='red', markersize=6),
# ]
#
# ax.legend(handles=legend_elements, loc="upper left")

plt.show()

In [ ]:
rain = wxt["R"].compute()

rain_events = rain.where(rain > 0, drop=True)
rain_p50 = rain_events.quantile(0.50)
rain_p95 = rain_events.quantile(0.95)

mask_rain = rain > 0
mask_p50 = rain > rain_p50
mask_p95 = rain > rain_p95

In [ ]:
years = np.unique(temperature['time'].dt.year)

n_years = len(years)
ncols = 4
nrows = int(np.ceil(n_years / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 6*nrows), gridspec_kw={"wspace": 0.2, "hspace": 0.35})
axes = axes.flatten()

for year, ax in zip(years, axes):
    rainfall = rain.sel(time=temperature["time"].dt.year == year)
    temps = temperature.sel(time=temperature["time"].dt.year == year)
    rel_hum = relative_humidity.sel(time=temperature["time"].dt.year == year)

    ax.scatter(temps, rel_hum, s=0.3, color='darkgray', alpha=0.4)
    ax.scatter(temps.where(rainfall > 0), rel_hum.where(rainfall > 0), s=0.5, color='#8000ff', alpha=0.75)
    ax.scatter(temps.where(rainfall > rain_p50), rel_hum.where(rainfall > 0), s=0.5, color='#FFAC1C', alpha=0.75)
    ax.scatter(temps.where(rainfall > rain_p95), rel_hum.where(rainfall > 0), s=1, color='red')

    ax.set_title(textwrap.fill(f"{year} relative humidity vs temperature plot (colored by rainfall intensity)", width=40))
    ax.set_ylabel("Relative Humidity / %")
    ax.set_xlabel("Temperature / °C")
    ax.set_xlim(18, 35)
    ax.set_ylim(40, 101)
    ax.set_xticks(np.arange(18, 35, 2))

for ax in axes[n_years:]:
    ax.set_visible(False)

plt.show()

Next, it makes sense to plot the relative humidity by percentiles in temperature bins.

When plotted by year, we see sme additional nuances.

1. Many of the high spikes in the dry relative humidity are concentrated in 2016.
2. 2021 also has 2 areas with unnatural vertical lines in the data.
3. 2025 has a great concentration of high relative humidity values near 100. Most other years are around the region between 90 and 95.

In [ ]:
temp_bins = np.arange(21, 33, 0.5)
binned = relative_humidity.groupby_bins(temperature, temp_bins)

rh_p95 = binned.quantile(0.95)
rh_p05 = binned.quantile(0.05)
rh_median = binned.quantile(0.5)

In [ ]:
temp_bins[:-1]

In [ ]:
counts = binned.count()

plt.plot(bin_centers, counts)
plt.xlabel("Temperature (°C)")
plt.ylabel("Number of observations")

### Problematic Area 1

In [ ]:
mask = (
    (relative_humidity >= 46) &
    (relative_humidity <= 63) &
    (temperature >= 18) &
    (temperature <= 23.5)
)

problem_area_1 = relative_humidity.where(mask, drop=True)

In [ ]:
problem_area_1["time"]

In [ ]:
fig, ax = plt.subplots(figsize=(10, 9))

ax.plot(relative_humidity["time"], relative_humidity)
ax.scatter(problem_area_1["time"], problem_area_1, color="red")

In [ ]:
rh_1 = relative_humidity.sel(time=slice("2011-03-01", "2013-09-01"))
prob_1 = problem_area_1.sel(time=slice("2011-03-01", "2013-09-01"))

rh_2 = relative_humidity.sel(time=slice("2015-10-23", "2015-10-24"))
prob_2 = problem_area_1.sel(time=slice("2015-10-23", "2015-10-24"))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12,8))

ax1.plot(rh_1["time"], rh_1)
ax1.scatter(prob_1["time"], prob_1, color="red")

ax2.plot(rh_2["time"], rh_2)
ax2.scatter(prob_2["time"], prob_2, color="red")

plt.show()

From the scatter plots, it appears that some smoothing in the regions of these problem areas as well as throughout the dataset can be applied for more realistic results.

In [ ]:
rain_events.to_series().describe()

# Verifying other Variables with Relative Humidity
The dataset does not have moisture measurements so we will derive them from relative humidity to ensure that the data is working as intended.

In [ ]:
rh = wxt["RH"].compute() / 100
p = wxt["P"].compute() * 100
t = wxt['T'].compute() + 273.15
q = relative_humidity_to_specific_humidity(rh, p, t)
e = specific_humidity_to_partial_pressure(q, p)
es = es(t)

In [ ]:
wxt['Q'] = q
wxt['E'] = e

In [ ]:
wxt

## Reconstructing RH from q

We will reconstruct the relative humidity from the specific humidity and compare it to the relative humidity measured from the instrument.

In [ ]:
rh_reconstructed = e / es * 100

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

ax.scatter(rh_reconstructed, relative_humidity)
ax.set_xlabel("Relative Humidity / %")
ax.set_ylabel("Reconstructed Relative Humidity / %")
ax.set_title("Reconstructed relative humidity vs instrument relative humidity")

plt.show()

The relative humidity is recreated very well.

## Specific humidity vs Temperature

In [ ]:
temperature_bins = np.arange(18, 33.1, 0.2)
q_binned = q.groupby_bins(temperature, temperature_bins)

q_p95 = q_binned.quantile(0.95)
q_p05 = q_binned.quantile(0.05)
q_median = q_binned.quantile(0.50)

In [ ]:
bin_centers = temperature_bins[:-1] + np.diff(temperature_bins)/2

fig, ax = plt.subplots(figsize=(10,6))

ax.plot(bin_centers, q_median, label="Median RH", color="black")
ax.plot(bin_centers, q_p95, label="95th percentile", color="red")
ax.plot(bin_centers, q_p05, label="5 th percentile", color="blue")

ax.fill_between(bin_centers, q_p05, q_p95, alpha=0.2)

ax.set_xlabel("Temperature / °C")
ax.set_ylabel("Specific Humidity / g/kg")
ax.set_title("Specific Humidity distribution as a function of temperature")
ax.set_xlim(22, 33)
ax.set_ylim(0.01, 0.02)

ax.legend()

plt.show()

## Dew Point
We will use Magnus' Formula to compute the dew point.

In [ ]:
def magnus(T: float, RH: float) -> float:
    """Returns the dew point temperature

    Args:
        T: Temperature in degrees Celsius
        RH: Relative Humidity as a fraction

    Returns:
        Dew Point in degrees Celsius
    """
    a = 17.625
    b = 243.04 # deg C

    gamma = (a * T) / (b + T) + np.log(RH)

    Td = (b * gamma) / (a - gamma)

    return Td

In [ ]:
Td = magnus(temperature, rh)

In [ ]:
wxt['Td'] = Td

In [ ]:
wxt

To reduce computational time, we resample the data into hourly data.

In [ ]:
hourly_temps = temperature.resample(time='1h').mean()
hourly_td = Td.resample(time='1h').mean()
hourly_rain = rain.resample(time='1h').sum()

hourly_rain_events = hourly_rain.where(hourly_rain > 0, drop=True)
hourly_rain_p50 = hourly_rain_events.quantile(0.50)
hourly_rain_p95 = hourly_rain_events.quantile(0.95)

# Voltage Supply

But we still need to look at each point so we also plot separate the plots by year.

In [ ]:
dew_point = wxt['Td'].compute()

In [ ]:
years = np.unique(temperature['time'].dt.year)

n_years = len(years)
ncols = 4
nrows = int(np.ceil(n_years / ncols))

fig, axes = plt.subplots(nrows, ncols, figsize=(6*ncols, 6*nrows), gridspec_kw={"wspace": 0.2, "hspace": 0.35}, dpi=300)
axes = axes.flatten()

for year, ax in zip(years, axes):
    rainfall = rain.sel(time=temperature["time"].dt.year == year)
    temps = temperature.sel(time=temperature["time"].dt.year == year)
    td = dew_point.sel(time=temperature["time"].dt.year == year)

    ax.scatter(temps, td, s=0.3, color='darkgray', alpha=0.4)
    ax.scatter(temps.where(rainfall > 0), td.where(rainfall > 0), s=0.5, color='#8000ff', alpha=0.75)
    ax.scatter(temps.where(rainfall > rain_p50), td.where(rainfall > 0), s=0.5, color='#FFAC1C', alpha=0.75)
    ax.scatter(temps.where(rainfall > rain_p95), td.where(rainfall > 0), s=1, color='red')

    ax.set_title(textwrap.fill(f"{year} Dew Point vs temperature plot (colored by rainfall intensity)", width=40))
    ax.set_ylabel("Dew Point / °C")
    ax.set_xlabel("Temperature / °C")
    ax.set_xlim(18, 35)
    ax.set_ylim(14, 30)
    ax.set_xticks(np.arange(18, 35, 2))

for ax in axes[n_years:]:
    ax.set_visible(False)

plt.show()

In [ ]:
voltage_supply = wxt['VS'].compute()
voltage_supply.plot()

In [ ]:
heater_voltage = wxt['VH'].compute()
heater_voltage.plot()

In [ ]:
ref_voltage = wxt['VR'].compute()
ref_voltage.plot()

In [ ]:
mask = (voltage_supply >= 8.5) | (voltage_supply <= 7.2)

voltage_spikes = voltage_supply.where(mask, drop=True)

voltage_spikes['time']

In [ ]:
corr_temp = xr.corr(temperature, voltage_supply, dim="time")
corr_rh   = xr.corr(relative_humidity, voltage_supply, dim="time")
corr_td   = xr.corr(dew_point, voltage_supply, dim="time")

In [ ]:
print(corr_temp, '\n')
print(corr_rh, '\n')
print(corr_td, '\n')

The correlation of around 0 for all variables means that there is not really a relationship between the voltage supply and the other variables.

In [ ]:
low_voltage  = voltage_supply < 7.9
normal_voltage = voltage_supply >= 7.9

In [ ]:
high_rh = relative_humidity > 95

prob_high_rh = high_rh.groupby_bins(voltage_supply, voltage_bins).mean()